In [1]:
import os
import glob
import numpy as np
from PIL import Image

# =========================================================================
# CONFIGURATION
# =========================================================================
DATA_DIR = "../"

TILE_ROWS = 32
TILE_COLS = 32
BUFFER_COLS = 40  # 40 tiles wide total hardware memory
BUFFER_ROWS = 30  # 30 tiles high total hardware memory

# What the PLAYER actually sees on screen at one time
VISIBLE_COLS = 20 # 20 tiles wide (640 pixels)
VISIBLE_ROWS = 15 # 15 tiles high (480 pixels) - Adjust this if your screen height is different!
# =========================================================================

def decode_7bit_to_rgba(tile_data):
    """Reverses the 7-bit packing logic to construct an 8-bit RGBA image."""
    rgba_img = np.zeros((TILE_ROWS, TILE_COLS, 4), dtype=np.uint8)
    
    # Extract bit components using bitwise masking
    a_1bit = (tile_data >> 6) & 0x1
    r_2bit = (tile_data >> 4) & 0x3
    g_2bit = (tile_data >> 2) & 0x3
    b_2bit = tile_data & 0x3
    
    # Upscale 2-bit (0-3) back to 8-bit (0-255) smoothly
    rgba_img[:, :, 0] = r_2bit * 85  # Red
    rgba_img[:, :, 1] = g_2bit * 85  # Green
    rgba_img[:, :, 2] = b_2bit * 85  # Blue
    
    # Handle alpha (1 = Transparent, 0 = Opaque)
    rgba_img[:, :, 3] = np.where(a_1bit == 1, 0, 255)
    
    return rgba_img

def load_tile_pixels(filepath):
    """Reads a tile's .mem file and returns a list of 1024 7-bit integers."""
    pixels = []
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if line:
                pixels.append(int(line, 2))
    return pixels

def main():
    print(f"Scanning directory: '{os.path.abspath(DATA_DIR)}' for .mem files...\n")

    # 1. Dynamically locate the backbuffer layout file
    buffer_search = glob.glob(os.path.join(DATA_DIR, "*backbuffer*.mem*"))
    if not buffer_search:
        print(f"❌ ERROR: Could not find your backbuffer layout file in '{DATA_DIR}'")
        print("Expected something like: backbuffer_init.mem")
        print("\nAvailable files in this directory:")
        for f in os.listdir(DATA_DIR)[:20]:
            print(f"  - {f}")
        print("\n👉 Please update the 'DATA_DIR' variable at the top of this script to point to the correct folder.")
        return
    
    buffer_file = buffer_search[0]
    print(f"Found layout file: {os.path.basename(buffer_file)}")

    # 2. Dynamically locate all background tile files
    tile_files = glob.glob(os.path.join(DATA_DIR, "*backtile_init_*.mem*"))
    if not tile_files:
        print(f"❌ ERROR: Found layout but couldn't find any tile files (e.g., backtile_init_0.mem) in '{DATA_DIR}'")
        return

    print(f"Found {len(tile_files)} tile files. Decoding...")
    tile_registry = {}
    for filepath in tile_files:
        try:
            filename = os.path.basename(filepath)
            # Safely parse out the tile number from filenames like 'backtile_init_0.mem'
            parts = filename.replace('.', '_').split('_')
            tile_id = None
            for p in reversed(parts):
                if p.isdigit():
                    tile_id = int(p)
                    break
            
            if tile_id is None:
                continue

            raw_pixels = load_tile_pixels(filepath)
            tile_array = np.array(raw_pixels).reshape((TILE_ROWS, TILE_COLS))
            rgba_tile = decode_7bit_to_rgba(tile_array)
            tile_registry[tile_id] = rgba_tile
        except Exception as e:
            print(f"  ⚠️ Warning: Skipping {filename} due to parsing error: {e}")

    print(f"Successfully cached {len(tile_registry)} unique tiles.")

    # 3. Read the 40x30 screen layout entries
    with open(buffer_file, 'r') as f:
        layout_indices = [int(line.strip(), 2) for line in f if line.strip()]

    if len(layout_indices) != BUFFER_COLS * BUFFER_ROWS:
        print(f"❌ ERROR: Layout file has {len(layout_indices)} items, but a 40x30 screen needs exactly {BUFFER_COLS * BUFFER_ROWS} items.")
        return

    # Reshape the 1D list into a 2D layout grid of (30 rows, 40 columns)
    layout_grid = np.array(layout_indices).reshape((BUFFER_ROWS, BUFFER_COLS))

    # 4. Initialize blank canvas for the entire background memory space (1280 x 960)
    full_width = BUFFER_COLS * TILE_COLS
    full_height = BUFFER_ROWS * TILE_ROWS
    full_canvas = np.zeros((full_height, full_width, 4), dtype=np.uint8)

    # 5. Draw the background tile-by-tile
    for row in range(BUFFER_ROWS):
        for col in range(BUFFER_COLS):
            tile_id = layout_grid[row, col]
            
            if tile_id in tile_registry:
                tile_rgba = tile_registry[tile_id]
            else:
                tile_rgba = np.zeros((TILE_ROWS, TILE_COLS, 4), dtype=np.uint8)
                tile_rgba[:, :, 3] = 255 # Opaque blank tile

            y_start = row * TILE_ROWS
            y_end = y_start + TILE_ROWS
            x_start = col * TILE_COLS
            x_end = x_start + TILE_COLS
            
            full_canvas[y_start:y_end, x_start:x_end] = tile_rgba

    # 6. Save the full horizontal scrolling map (keeps all 30 rows and 40 columns)
    full_image = Image.fromarray(full_canvas, 'RGBA')
    full_image.save("full_background_buffer.png")
    print("\n✅ Saved complete memory map layout to 'full_background_buffer.png' (1280x960)")

    # 7. Extract and save just what is inside the player's viewport (X and Y axes cropped)
    visible_width = VISIBLE_COLS * TILE_COLS
    visible_height = VISIBLE_ROWS * TILE_ROWS
    
    # Slices both dimensions: [:visible_height] trims the bottom, [:visible_width] trims the right
    visible_canvas = full_canvas[:visible_height, :visible_width] 
    
    visible_image = Image.fromarray(visible_canvas, 'RGBA')
    visible_image.save("visible_screen.png")
    print(f"✅ Saved visible screen space to 'visible_screen.png' ({visible_width}x{visible_height})")

if __name__ == "__main__":
    main()

Scanning directory: 'c:\02113 Game\02113-game-project\memory_init' for .mem files...

Found layout file: backbuffer_init.mem
Found 32 tile files. Decoding...
Successfully cached 32 unique tiles.

✅ Saved complete memory map layout to 'full_background_buffer.png' (1280x960)
✅ Saved visible screen space to 'visible_screen.png' (640x480)


C:\Users\mikke\AppData\Local\Temp\ipykernel_46624\2365302962.py:134: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  full_image = Image.fromarray(full_canvas, 'RGBA')
C:\Users\mikke\AppData\Local\Temp\ipykernel_46624\2365302962.py:145: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  visible_image = Image.fromarray(visible_canvas, 'RGBA')
